# Modify: representación y aislamiento

Laboratorio ESPE · Semilla 42 · Fuentes y límites documentados.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / 'data' / 'dataset.csv').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image
df = pd.read_csv(ROOT / 'data/dataset.csv', keep_default_na=False)
df['labels'] = df.labels.map(json.loads)
print('Raíz del laboratorio:', ROOT)


Raíz del laboratorio: D:\ESPE\36902-DESARROLLO SOFTWARE SEGURO\Pacial2\Trabajo


Objetivo: obtener matrices TF-IDF sin ajustar el vocabulario con validación o prueba. Se conservan constantes de seguridad como False, true y md5. No se usa scenario, CWE, project ni labels como característica. El clasificador no ve metadatos de etiquetado.

In [2]:
from vulnminer.features import make_features, normalize
from sklearn.preprocessing import MultiLabelBinarizer
parts = {s: df[df.split == s] for s in ['train','validation','test']}
features = make_features()
X_train = features.fit_transform(parts['train'].code)
X_validation = features.transform(parts['validation'].code)
X_test = features.transform(parts['test'].code)
mlb = MultiLabelBinarizer()
y_train = mlb.fit_transform(parts['train'].labels)
print('Matrices:', X_train.shape, X_validation.shape, X_test.shape)
print('Etiquetas:', mlb.classes_.tolist())
print('Normalización:', normalize('# comment\napp.run(debug=True)'))

Matrices: (672, 7884) (168, 7884) (168, 7884)
Etiquetas: ['A01:2025', 'A02:2025', 'A04:2025', 'A05:2025', 'A07:2025', 'A08:2025', 'API10:2023', 'API2:2023', 'API7:2023', 'API8:2023']
Normalización: app.run(debug=True)


In [3]:
from scipy.sparse import load_npz
for split in parts:
    saved = load_npz(ROOT / 'output/evaluation' / f'features_{split}.npz')
    print(split, 'matriz entregada', saved.shape, 'no ceros', saved.nnz)
canonical_stats = json.loads((ROOT / 'output/evaluation/dataset_statistics.json').read_text())
print('Duplicados canónicos entre particiones:', canonical_stats['canonical_cross_partition_duplicates'])

train matriz entregada (672, 7884) no ceros 146523
validation matriz entregada (168, 7884) no ceros 33648
test matriz entregada (168, 7884) no ceros 29732
Duplicados canónicos entre particiones: 0


Justificación: los n-gramas de caracteres 3..5 toleran cambios de identificadores; los tokens 1..2 retienen operaciones y constantes. Se eliminan comentarios de línea y se normalizan saltos de línea, sin reemplazar constantes relevantes. Los datos ya generados pasan controles de duplicados e integridad antes de transformar: si fallan se interrumpe el entrenamiento. Ambos algoritmos usan ponderación de clases; no se requiere sobremuestreo. La auditoría canónica solo cubre sustituciones documentadas de dominios y puertos.